# Experiment

## Import libraries

In [1]:
import pandas as pd

file_path = "credit_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="month_str",
    value_name="value",
)

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and month
df["date"] = pd.to_datetime(df["month_str"], errors="coerce")

# Drop rows where date couldn't be parsed
df = df.dropna(subset=["date"])

# Extract numeric year, month
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "month"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by year and month
df = df.sort_values(["year", "month"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22884\2172693298.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["month_str"], errors="coerce")


Chỉ tiêu,year,month,credit,credit_growth_ytd,money_supply_growth_m2_ytd,money_supply_m2
0,2009,3,1351631.00,6.00,7.77,1748226.0
1,2009,6,1531107.00,20.08,17.59,1907523.0
2,2009,9,1667854.00,30.80,22.43,1986084.0
3,2009,12,1753600.00,37.53,28.99,2092447.0
4,2010,1,1938558.00,0.26,0.40,1951164.0
...,...,...,...,...,...,...
182,2025,1,15701965.17,0.55,1.46,18176926.0
183,2025,2,15735537.26,0.76,1.35,18157191.0
184,2025,3,16226458.02,3.91,2.99,18450042.0
185,2025,4,16446560.51,5.32,4.50,18721571.0


In [2]:
df.columns

Index(['year', 'month', 'credit', 'credit_growth_ytd',
       'money_supply_growth_m2_ytd', 'money_supply_m2'],
      dtype='object', name='Chỉ tiêu')